# V5 DINOv2 with LoRA + Dual Projection Heads Training on BigEarthNet-14k

This notebook contains the complete, self-contained training pipeline for the new **V5 model** using **DINOv2 (`facebook/dinov2-base`)** with **LoRA** and **Dual Projection Heads** trained on the **BigEarthNet-14k** dataset.

### Pipeline Overview:
1. **Environment Setup**: Install PEFT, Transformers, Rasterio, Kagglehub, etc.
2. **Dataset Download**: Automatically authenticates and downloads BigEarthNet-14k from Kaggle.
3. **Preprocessing**: Extracts and converts all 14k image pairs into pre-processed `.npy` files to ensure high-speed training without CPU/IO bottlenecks.
4. **LoRA Fine-tuning**: Fine-tunes attention layers (`query`, `value`) of DINOv2 with contrastive InfoNCE loss.
5. **Dual Projections**: Maps 768-dim optical & SAR features into a shared 256-dim space.
6. **FAISS Indexing**: Builds a similarity index of the trained SAR gallery.
7. **Artifact Export**: Packages model weights (`opt_proj.pt`, `sar_proj.pt`, `lora_adapter/`) for deployment.

## 1. Setup & Installations

In [ ]:
!pip install -q peft transformers rasterio scipy pandas numpy torch tifffile accelerate kagglehub faiss-cpu

## 2. Authenticate Kaggle & Download Dataset

In [ ]:
import os
import kagglehub

# Set Kaggle API credentials
os.environ["KAGGLE_USERNAME"] = "kgat"
os.environ["KAGGLE_KEY"] = "041ab2df77bab20fd0a79c8d56d31814"

print("Downloading bigearthnet-14k dataset using kagglehub...")
dataset_path = kagglehub.dataset_download("narendraaironi/bigearthnet-14k")
print("Path to dataset files:", dataset_path)

## 3. Preprocessing Pipeline

We will preprocess the TIFF files to `.npy` arrays beforehand. This avoids expensive CPU resizing (`scipy.ndimage.zoom`) and TIFF parsing (`rasterio`) during training.

In [ ]:
import numpy as np
import pandas as pd
import rasterio
from scipy.ndimage import zoom as scipy_zoom
from pathlib import Path
from tqdm.notebook import tqdm

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)
S2_RGB_INDICES = [2, 1, 0]
S2_REFLECTANCE_SCALE = 10000.0
TARGET_SIZE = 224

def resize_to_224(arr, target=TARGET_SIZE):
    _, h, w = arr.shape
    if h == target and w == target:
        return arr
    return scipy_zoom(arr, (1, target / h, target / w), order=1).astype(arr.dtype)

def preprocess_optical(path):
    with rasterio.open(path) as src:
        data = src.read()  # (C, H, W)
    rgb = data[S2_RGB_INDICES].astype(np.float32)
    rgb = resize_to_224(rgb, TARGET_SIZE)
    rgb /= S2_REFLECTANCE_SCALE
    np.clip(rgb, 0.0, 1.0, out=rgb)
    for c in range(3):
        rgb[c] = (rgb[c] - IMAGENET_MEAN[c]) / IMAGENET_STD[c]
    return rgb

def preprocess_sar(path):
    with rasterio.open(path) as src:
        data = src.read()  # (2, H, W)
    vv = data[0].astype(np.float32)
    vh = data[1].astype(np.float32)
    sar_3ch = np.stack([vv, vh, vv - vh], axis=0)
    sar_3ch = resize_to_224(sar_3ch, TARGET_SIZE)
    return sar_3ch

In [ ]:
kaggle_root = Path(dataset_path) / "BEN_14k"
metadata_path = kaggle_root / "metadata.parquet"
df = pd.read_parquet(metadata_path)
print(f"Loaded metadata parquet. Total rows: {len(df)}")

opt_out_dir = Path("/content/processed/optical")
sar_out_dir = Path("/content/processed/sar")
opt_out_dir.mkdir(parents=True, exist_ok=True)
sar_out_dir.mkdir(parents=True, exist_ok=True)

# Preprocess all pairs and save as .npy arrays
processed_rows = []

for split in ["train", "validation", "test"]:
    split_df = df[df["split"] == split]
    print(f"Preprocessing {split} split ({len(split_df)} candidate rows)...")
    s1_dir = kaggle_root / "BigEarthNet-S1" / split
    s2_dir = kaggle_root / "BigEarthNet-S2" / split
    
    for _, row in tqdm(split_df.iterrows(), total=len(split_df)):
        patch_id = row["patch_id"]
        s1_name = row["s1_name"]
        
        s1_path = s1_dir / f"{s1_name}.tif"
        s2_path = s2_dir / f"{patch_id}.tif"
        
        if s1_path.exists() and s2_path.exists():
            try:
                opt_npy = opt_out_dir / f"{patch_id}.npy"
                sar_npy = sar_out_dir / f"{patch_id}.npy"
                
                if not opt_npy.exists():
                    opt_arr = preprocess_optical(s2_path)
                    np.save(opt_npy, opt_arr)
                if not sar_npy.exists():
                    sar_arr = preprocess_sar(s1_path)
                    np.save(sar_npy, sar_arr)
                
                processed_rows.append({
                    "id": patch_id,
                    "optical_path": f"optical/{patch_id}.npy",
                    "sar_path": f"sar/{patch_id}.npy",
                    "split": split
                })
            except Exception as e:
                print(f"Error processing {patch_id}: {e}")

processed_df = pd.DataFrame(processed_rows)
processed_df.to_csv("/content/processed/metadata.csv", index=False)
print(f"Preprocessing complete! Preprocessed metadata saved to /content/processed/metadata.csv (Total: {len(processed_df)} pairs)")

## 4. PyTorch Dataset & Model Definitions

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel
from peft import LoraConfig, get_peft_model

class PreprocessedDataset(Dataset):
    def __init__(self, metadata_csv, split_name):
        df = pd.read_csv(metadata_csv)
        self.df = df[df["split"] == split_name].reset_index(drop=True)
        self.root = Path("/content/processed")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        opt = np.load(self.root / row["optical_path"])
        sar = np.load(self.root / row["sar_path"])
        return {
            "id": str(row["id"]),
            "opt": torch.tensor(opt, dtype=torch.float32),
            "sar": torch.tensor(sar, dtype=torch.float32)
        }

class ProjectionHead(nn.Module):
    def __init__(self, input_dim=768, output_dim=256, dropout_prob=0.4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.LayerNorm(512),
            nn.ReLU(),
            nn.Dropout(dropout_prob),
            nn.Linear(512, output_dim)
        )

    def forward(self, x):
        if self.training:
            x = x + torch.randn_like(x) * 0.02
        return F.normalize(self.net(x), p=2, dim=-1)

def clip_loss(opt_proj, sar_proj, temp=0.07):
    logits = torch.matmul(opt_proj, sar_proj.t()) / temp
    labels = torch.arange(opt_proj.size(0), device=opt_proj.device)
    return (F.cross_entropy(logits, labels) + F.cross_entropy(logits.t(), labels)) / 2

def encode_image(model, images):
    outputs = model(pixel_values=images)
    embeddings = outputs.last_hidden_state.mean(dim=1)
    return F.normalize(embeddings, p=2, dim=-1)

def retrieval_metrics(query_feat, gallery_feat):
    scores = np.matmul(query_feat, gallery_feat.T)
    n = scores.shape[0]
    top1 = top5 = top10 = 0
    for i in range(n):
        ranked = np.argsort(-scores[i])
        if i == ranked[0]:
            top1 += 1
        if i in ranked[:5]:
            top5 += 1
        if i in ranked[:10]:
            top10 += 1
    return {
        "recall@1": top1 / n * 100,
        "recall@5": top5 / n * 100,
        "recall@10": top10 / n * 100
    }

## 5. Training Loop (DINOv2 LoRA + Projection Heads)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 1. Initialize Datasets
train_ds = PreprocessedDataset("/content/processed/metadata.csv", "train")
val_ds = PreprocessedDataset("/content/processed/metadata.csv", "validation")
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=128, shuffle=False)

# 2. Load Model & Setup LoRA
base_model = AutoModel.from_pretrained("facebook/dinov2-base")
for param in base_model.parameters():
    param.requires_grad = False

peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["query", "value"],
    bias="none"
)
model = get_peft_model(base_model, peft_config)
model.to(device)

opt_head = ProjectionHead(input_dim=768, output_dim=256).to(device)
sar_head = ProjectionHead(input_dim=768, output_dim=256).to(device)

# Show trainable params
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad) + sum(p.numel() for p in opt_head.parameters() if p.requires_grad) + sum(p.numel() for p in sar_head.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters()) + sum(p.numel() for p in opt_head.parameters()) + sum(p.numel() for p in sar_head.parameters())
print(f"Trainable parameters: {trainable:,} / {total:,} ({trainable/total*100:.4f}%)")

# 3. Optimizer
optimizer = torch.optim.AdamW(
    [p for p in list(model.parameters()) + list(opt_head.parameters()) + list(sar_head.parameters()) if p.requires_grad],
    lr=0.0005,
    weight_decay=0.001
)

epochs = 15
best_val = -1.0
os.makedirs("/content/models/dinov2_lora", exist_ok=True)

# 4. Loop
for epoch in range(1, epochs + 1):
    model.train()
    opt_head.train()
    sar_head.train()
    losses = []
    
    for batch in tqdm(train_loader, desc=f"Epoch {epoch}/{epochs}"):
        opt = batch["opt"].to(device)
        sar = batch["sar"].to(device)
        
        optimizer.zero_grad()
        opt_proj = opt_head(encode_image(model, opt))
        sar_proj = sar_head(encode_image(model, sar))
        loss = clip_loss(opt_proj, sar_proj)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
        
    # Evaluation
    model.eval()
    opt_head.eval()
    sar_head.eval()
    opt_feats, sar_feats = [], []
    with torch.no_grad():
        for batch in val_loader:
            opt = batch["opt"].to(device)
            sar = batch["sar"].to(device)
            opt_feats.append(opt_head(encode_image(model, opt)).cpu().numpy())
            sar_feats.append(sar_head(encode_image(model, sar)).cpu().numpy())
            
    metrics = retrieval_metrics(np.vstack(opt_feats), np.vstack(sar_feats))
    avg_loss = np.mean(losses)
    print(f"Epoch {epoch:02d} | Loss: {avg_loss:.4f} | Val R@1: {metrics['recall@1']:.2f}% | R@5: {metrics['recall@5']:.2f}%")
    
    if metrics["recall@1"] > best_val:
        best_val = metrics["recall@1"]
        torch.save(opt_head.state_dict(), "/content/models/dinov2_lora/opt_proj.pt")
        torch.save(sar_head.state_dict(), "/content/models/dinov2_lora/sar_proj.pt")
        model.save_pretrained("/content/models/dinov2_lora/lora_adapter")
        print(f" Saved new best weights with Recall@1: {best_val:.2f}%")

## 6. Build FAISS Index for the Gallery

In [ ]:
import faiss

print("Generating V5 FAISS Index...")
model.eval()
sar_head.eval()

test_ds = PreprocessedDataset("/content/processed/metadata.csv", "test")
test_loader = DataLoader(test_ds, batch_size=128, shuffle=False)

sar_feats = []
sar_names = []

with torch.no_grad():
    for batch in test_loader:
        sar = batch["sar"].to(device)
        sar_e = encode_image(model, sar)
        proj = sar_head(sar_e).cpu().numpy()
        sar_feats.append(proj)
        sar_names.extend(batch["id"])

sar_feats = np.vstack(sar_feats).astype("float32")
sar_feats = sar_feats / np.linalg.norm(sar_feats, axis=1, keepdims=True)

index = faiss.IndexFlatIP(sar_feats.shape[1])
index.add(sar_feats)

faiss.write_index(index, "/content/models/dinov2_lora/dinov2_lora_projected_gallery.index")
np.save("/content/models/dinov2_lora/dinov2_lora_gallery_names.npy", np.array(sar_names))

print("Index creation completed successfully!")

## 7. Package and Export Weights

In [ ]:
import shutil

shutil.make_archive("/content/dinov2_lora_v5_weights", 'zip', "/content/models/dinov2_lora")
print("Artifacts zipped to /content/dinov2_lora_v5_weights.zip")
print("You can download this zip file directly from Google Colab and place the contents in models/dinov2_lora/ directory on your local workspace.")